# Previsão da 1ª Rodada da Copa do Mundo 2026
### Deep Learning — Turma 1/2026 | Eliane Sa Ricarte

---

## 1. Introdução

Este notebook implementa um pipeline completo de Machine Learning para prever os placares dos 24 jogos da primeira rodada da fase de grupos da Copa do Mundo FIFA 2026, utilizando TensorFlow/Keras.

### Hipótese central
A força histórica relativa entre duas seleções — capturada pelo rating ELO — combinada com a forma recente de cada equipe e a importância do torneio em que jogaram, contém informação suficiente para prever o placar esperado de um jogo internacional de futebol.

### Abordagem
Modelamos o problema como uma **regressão de Poisson dupla**: o modelo prevê independentemente a taxa esperada de gols (λ) de cada seleção, e o placar final é obtido arredondando essas taxas para o inteiro mais próximo. Essa abordagem é amplamente utilizada na literatura de previsão esportiva por respeitar a natureza discreta e assimétrica da distribuição de gols no futebol.

### Dependências
- Python 3.11.15
- TensorFlow 2.19.0
- pandas, numpy, scikit-learn
- duckdb (para leitura do Transfermarkt dataset)

In [55]:
import pandas as pd

# Carregar as bases
results = pd.read_csv('Resultados_Internacionais_1872_2026.csv')
rankings = pd.read_csv('FIFA_World_Rankings_1992_2024.csv')
copa_america = pd.read_csv('Copa_America_Historico_desde_1916.csv')
odds = pd.read_csv('WC2026_ODS.csv')

## 2. Dados

### Fontes utilizadas

| Base | Fonte | Período | Conteúdo |
|---|---|---|---|
| Resultados Internacionais | Kaggle (martj42) | 1872–2026 | 49.287 partidas internacionais |
| FIFA World Rankings | Kaggle (cashncarry) | 1992–2024 | Rankings mensais por seleção |
| Rankings + Partidas Cruzados | Kaggle (gabipana7) | 1992–2022 | Rankings cruzados com resultados |
| Copa América Histórico | Kaggle | 1916–2024 | Estatísticas da Copa América |
| Transfermarkt Datasets | GitHub (dcaribou) | Atualizado | World Cup, Euro, Copa América, AFCON, Asian Cup |
| Odds WC2026 | Oddschecker (coleta manual) | Mai/2026 | Odds 1X2 dos 24 jogos da 1ª rodada |

### Base principal: Resultados Internacionais (1872–2026)
A base `results.csv` é o backbone do projeto. Contém data, times, placar, torneio, cidade e se o jogo foi em campo neutro. A coluna `tournament` permite filtrar por tipo de competição, aplicando pesos diferentes conforme a importância do jogo.

### Cobertura das 48 seleções
Todas as 48 seleções participantes da Copa 2026 estão presentes na base, com volume mínimo de 27 jog

In [56]:
# Visão geral de cada base
print("=== RESULTADOS INTERNACIONAIS ===")
print(f"Shape: {results.shape}")
print(results.dtypes)
print(results.head(3))
print()

print("=== FIFA RANKINGS ===")
print(f"Shape: {rankings.shape}")
print(rankings.dtypes)
print(rankings.head(3))
print()

print("=== COPA AMÉRICA ===")
print(f"Shape: {copa_america.shape}")
print(copa_america.dtypes)
print(copa_america.head(3))
print()

print("=== ODDS WC2026 ===")
print(f"Shape: {odds.shape}")
print(odds.head(3))

=== RESULTADOS INTERNACIONAIS ===
Shape: (49287, 9)
date              str
home_team         str
away_team         str
home_score    float64
away_score    float64
tournament        str
city              str
country           str
neutral          bool
dtype: object
         date home_team away_team  home_score  away_score tournament     city  \
0  1872-11-30  Scotland   England         0.0         0.0   Friendly  Glasgow   
1  1873-03-08   England  Scotland         4.0         2.0   Friendly   London   
2  1874-03-07  Scotland   England         2.0         1.0   Friendly  Glasgow   

    country  neutral  
0  Scotland    False  
1   England    False  
2  Scotland    False  

=== FIFA RANKINGS ===
Shape: (67472, 8)
rank               float64
country_full           str
country_abrv           str
total_points       float64
previous_points    float64
rank_change          int64
confederation          str
rank_date              str
dtype: object
    rank       country_full country_abrv  total_

## 3. Pré-processamento

### 3.1 Pesos por tipo de torneio

Nem todos os jogos têm o mesmo valor informativo. Uma vitória na Copa do Mundo reflete muito mais a força real de uma seleção do que um amistoso de preparação. Por isso, atribuímos pesos diferentes a cada tipo de torneio:

| Peso | Torneios |
|---|---|
| 1.0 | FIFA World Cup |
| 0.9 | Copa América, AFCON, UEFA Euro, AFC Asian Cup |
| 0.8 | Eliminatórias de todas as confederações, Gold Cup |
| 0.7 | UEFA Nations League, CONCACAF Nations League |
| 0.3 | Amistosos e demais torneios |

Esses pesos são usados tanto no cálculo do ELO quanto como feature direta do modelo.

### 3.2 Rating ELO

O ELO é um sistema de pontuação que avalia a força de cada seleção com base no histórico de resultados. Originalmente criado para o xadrez, é amplamente utilizado no futebol por capturar a força relativa entre equipes de forma dinâmica.

**Funcionamento:**
- Todas as seleções começam com **1500 pontos**
- Após cada jogo, o ELO é atualizado c

In [57]:
# As 48 seleções da Copa 2026
selecoes_2026 = [
    'Mexico','South Africa','South Korea','Czech Republic',
    'Canada','Bosnia and Herzegovina','Qatar','Switzerland',
    'Brazil','Morocco','Haiti','Scotland',
    'United States','Paraguay','Australia','Turkey',
    'Germany','Curacao','Ivory Coast','Ecuador',
    'Netherlands','Japan','Sweden','Tunisia',
    'Belgium','Egypt','Iran','New Zealand',
    'Spain','Cape Verde','Saudi Arabia','Uruguay',
    'France','Senegal','Iraq','Norway',
    'Argentina','Algeria','Austria','Jordan',
    'Portugal','DR Congo','Uzbekistan','Colombia',
    'England','Croatia','Ghana','Panama'
]

# Verificar cobertura na base de resultados
todos_times = set(results['home_team']).union(set(results['away_team']))
sem_dados = [s for s in selecoes_2026 if s not in todos_times]

print(f"Seleções com dados: {len(selecoes_2026) - len(sem_dados)}/48")
print(f"\nSeleções SEM dados na base:")
for s in sem_dados:
    print(f"  ❌ {s}")

Seleções com dados: 47/48

Seleções SEM dados na base:
  ❌ Curacao


In [58]:
# Buscar como Curaçao aparece na base
times = list(todos_times)
curaçao_variantes = [t for t in times if 'ur' in t.lower() and ('a' in t.lower())]
print(curaçao_variantes)

['Mauritania', 'Arameans Suryoye', 'Maule Sur', 'Uruguay', 'Honduras', 'Burkina Faso', 'Turkmenistan', 'Asturias', 'Mauritius', 'Curaçao', 'Suriname', 'Iraqi Kurdistan', 'Turks and Caicos Islands', 'Darfur']


In [59]:
# Corrigir o nome
selecoes_2026 = [s if s != 'Curacao' else 'Curaçao' for s in selecoes_2026]

# Verificar novamente
sem_dados = [s for s in selecoes_2026 if s not in todos_times]
print(f"Seleções com dados: {len(selecoes_2026) - len(sem_dados)}/48")
print(f"Seleções SEM dados: {sem_dados}")

Seleções com dados: 48/48
Seleções SEM dados: []


In [60]:
# Contar jogos por seleção
jogos_por_selecao = {}
for selecao in selecoes_2026:
    total = len(results[(results['home_team'] == selecao) | 
                        (results['away_team'] == selecao)])
    jogos_por_selecao[selecao] = total

# Ordenar do menor para o maior
jogos_df = pd.DataFrame.from_dict(jogos_por_selecao, orient='index', 
                                   columns=['total_jogos']).sort_values('total_jogos')

print("=== SELEÇÕES COM MENOS DADOS ===")
print(jogos_df.head(10))
print()
print("=== SELEÇÕES COM MAIS DADOS ===")
print(jogos_df.tail(10))

=== SELEÇÕES COM MENOS DADOS ===
                        total_jogos
Cape Verde                      236
Bosnia and Herzegovina          284
Uzbekistan                      355
Czech Republic                  361
Curaçao                         386
Croatia                         397
New Zealand                     417
Canada                          470
South Africa                    480
Jordan                          482

=== SELEÇÕES COM MAIS DADOS ===
             total_jogos
Switzerland          885
France               936
Uruguay              971
Mexico              1003
South Korea         1008
Germany             1032
Brazil              1060
Argentina           1067
England             1091
Sweden              1102


In [61]:
# Verificar jogos OFICIAIS e RECENTES (últimos 4 anos) por seleção
results['date'] = pd.to_datetime(results['date'])

# Filtrar só jogos oficiais (excluindo amistosos) e recentes
oficiais_recentes = results[
    (results['tournament'] != 'Friendly') &
    (results['date'] >= '2022-01-01')
]

jogos_oficiais = {}
for selecao in selecoes_2026:
    total = len(oficiais_recentes[
        (oficiais_recentes['home_team'] == selecao) | 
        (oficiais_recentes['away_team'] == selecao)
    ])
    jogos_oficiais[selecao] = total

oficiais_df = pd.DataFrame.from_dict(jogos_oficiais, orient='index',
                                      columns=['jogos_oficiais_recentes']).sort_values('jogos_oficiais_recentes')

print("=== SELEÇÕES COM MENOS JOGOS OFICIAIS RECENTES (2022-2026) ===")
print(oficiais_df.head(15))

=== SELEÇÕES COM MENOS JOGOS OFICIAIS RECENTES (2022-2026) ===
                        jogos_oficiais_recentes
New Zealand                                  27
Paraguay                                     29
Colombia                                     31
Norway                                       31
Curaçao                                      33
Sweden                                       33
Germany                                      33
Ecuador                                      33
Uruguay                                      34
Scotland                                     35
Brazil                                       35
Haiti                                        35
South Korea                                  36
Bosnia and Herzegovina                       36
Czech Republic                               37


In [62]:
print("=== TIPOS DE TORNEIO NA BASE ===")
print(results['tournament'].value_counts().head(20))

=== TIPOS DE TORNEIO NA BASE ===
tournament
Friendly                                18252
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1036
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620
CFU Caribbean Cup qualification           606
Merdeka Tournament                        599
British Home Championship                 523
CONCACAF Nations League                   422
AFC Asian Cup                             421
Gold Cup                                  420
Gulf Cup                                  410
Island Games                              394
UEFA Euro                                 388
Asian Games                               368
Name: count, dtype: int64


In [63]:
import sys
sys.path.append('.')
from Mapeamento_Selecoes import nome_para_sigla, sigla_para_nome

# Pesos por tipo de torneio
pesos_torneio = {
    'FIFA World Cup':                      1.0,
    'Copa América':                        0.9,
    'African Cup of Nations':              0.9,
    'UEFA Euro':                           0.9,
    'AFC Asian Cup':                       0.9,
    'FIFA World Cup qualification':        0.8,
    'UEFA Euro qualification':             0.8,
    'African Cup of Nations qualification':0.8,
    'AFC Asian Cup qualification':         0.8,
    'CONCACAF Championship':               0.8,
    'Gold Cup':                            0.8,
    'UEFA Nations League':                 0.7,
    'CONCACAF Nations League':             0.7,
}
# Amistosos e todo o resto recebem 0.3
peso_padrao = 0.3

results['peso'] = results['tournament'].map(pesos_torneio).fillna(peso_padrao)

print("Distribuição de pesos:")
print(results.groupby('peso')['tournament'].count().sort_index(ascending=False))

Distribuição de pesos:
peso
1.0     1036
0.9     2523
0.8    15340
0.7     1080
0.3    29308
Name: tournament, dtype: int64


In [64]:
import numpy as np

# Calcular ELO histórico para todas as seleções
K = 30  # fator de atualização
elo_ratings = {}

# Ordenar por data
results_sorted = results.sort_values('date').reset_index(drop=True)

def expected(ra, rb):
    return 1 / (1 + 10 ** ((rb - ra) / 400))

def atualizar_elo(home, away, home_score, away_score, peso, k=K):
    ra = elo_ratings.get(home, 1500)
    rb = elo_ratings.get(away, 1500)
    ea = expected(ra, rb)
    
    if home_score > away_score:
        sa, sb = 1, 0
    elif home_score < away_score:
        sa, sb = 0, 1
    else:
        sa, sb = 0.5, 0.5
    
    elo_ratings[home] = ra + k * peso * (sa - ea)
    elo_ratings[away]  = rb + k * peso * (sb - (1 - ea))

# Calcular ELO para cada jogo e salvar o ELO no momento do jogo
elo_home_list, elo_away_list = [], []

for _, row in results_sorted.iterrows():
    home, away = row['home_team'], row['away_team']
    elo_home_list.append(elo_ratings.get(home, 1500))
    elo_away_list.append(elo_ratings.get(away, 1500))
    atualizar_elo(home, away, row['home_score'], row['away_score'], row['peso'])

results_sorted['elo_home'] = elo_home_list
results_sorted['elo_away'] = elo_away_list
results_sorted['elo_diff'] = results_sorted['elo_home'] - results_sorted['elo_away']

# Ver ELO atual das 48 seleções
elo_copa = {sigla: elo_ratings.get(sigla_para_nome[sigla], 1500) 
            for sigla in sigla_para_nome}
elo_df = pd.DataFrame.from_dict(elo_copa, orient='index', 
                                 columns=['elo']).sort_values('elo', ascending=False)
print("=== ELO ATUAL DAS 48 SELEÇÕES ===")
print(elo_df.to_string())

=== ELO ATUAL DAS 48 SELEÇÕES ===
             elo
ESP  1941.721569
ARG  1915.965820
FRA  1905.466756
ENG  1863.225009
BRA  1850.813242
POR  1844.816030
NED  1834.359050
GER  1832.393880
JPN  1823.387683
MAR  1822.966360
COL  1815.257060
SEN  1797.486190
CRO  1797.165519
IRN  1791.782373
MEX  1788.988546
BEL  1788.536195
URU  1786.517761
ECU  1776.726867
KOR  1772.395618
AUS  1771.870520
USA  1764.278119
TUR  1757.653275
SUI  1751.291283
ALG  1748.125854
UZB  1739.474077
NOR  1735.748411
EGY  1733.112832
PAN  1723.958103
AUT  1720.574194
CAN  1718.275331
CIV  1715.370896
PAR  1712.979815
IRQ  1706.431400
SCO  1693.371563
TUN  1681.469883
SWE  1680.702779
JOR  1679.741323
KSA  1673.404146
CZE  1671.085277
COD  1669.713615
NZL  1665.448633
QAT  1657.951400
RSA  1646.604084
HAI  1612.844924
CPV  1609.198071
GHA  1609.157553
BIH  1582.465203
CUW  1556.779628


In [65]:
# Forma recente: média ponderada dos últimos N jogos de cada seleção
N_JOGOS = 10  # janela de jogos recentes

def calcular_forma(df, team, data_ref, n=N_JOGOS):
    """Retorna estatísticas dos últimos N jogos de uma seleção antes de data_ref"""
    jogos = df[
        ((df['home_team'] == team) | (df['away_team'] == team)) &
        (df['date'] < data_ref)
    ].tail(n)
    
    if len(jogos) == 0:
        return {'gols_marcados': 0, 'gols_sofridos': 0, 'pontos': 0, 'jogos': 0}
    
    gols_marcados, gols_sofridos, pontos = [], [], []
    
    for _, row in jogos.iterrows():
        if row['home_team'] == team:
            gm, gs = row['home_score'], row['away_score']
        else:
            gm, gs = row['away_score'], row['home_score']
        
        gols_marcados.append(gm)
        gols_sofridos.append(gs)
        
        if gm > gs:   pontos.append(3)
        elif gm == gs: pontos.append(1)
        else:          pontos.append(0)
    
    return {
        'gols_marcados': np.mean(gols_marcados),
        'gols_sofridos': np.mean(gols_sofridos),
        'pontos':        np.mean(pontos),
        'jogos':         len(jogos)
    }

# Testar com o Brasil
data_copa = pd.Timestamp('2026-06-11')
forma_bra = calcular_forma(results_sorted, 'Brazil', data_copa)
forma_mar = calcular_forma(results_sorted, 'Morocco', data_copa)

print("=== FORMA RECENTE (últimos 10 jogos) ===")
print(f"\nBrasil:")
print(f"  Gols marcados/jogo: {forma_bra['gols_marcados']:.2f}")
print(f"  Gols sofridos/jogo: {forma_bra['gols_sofridos']:.2f}")
print(f"  Pontos/jogo:        {forma_bra['pontos']:.2f}")

print(f"\nMarrocos:")
print(f"  Gols marcados/jogo: {forma_mar['gols_marcados']:.2f}")
print(f"  Gols sofridos/jogo: {forma_mar['gols_sofridos']:.2f}")
print(f"  Pontos/jogo:        {forma_mar['pontos']:.2f}")

=== FORMA RECENTE (últimos 10 jogos) ===

Brasil:
  Gols marcados/jogo: 1.80
  Gols sofridos/jogo: 0.80
  Pontos/jogo:        1.70

Marrocos:
  Gols marcados/jogo: 1.80
  Gols sofridos/jogo: 0.50
  Pontos/jogo:        2.40


In [66]:
# Montar dataset de treino completo
# Usar apenas jogos a partir de 2000 para ter ELO já calibrado
treino_data = results_sorted[results_sorted['date'] >= '2000-01-01'].copy()
treino_data = treino_data.dropna(subset=['home_score', 'away_score'])

print(f"Jogos para treino: {len(treino_data)}")
print(f"Período: {treino_data['date'].min()} a {treino_data['date'].max()}")

# Calcular forma recente para cada jogo do dataset
print("\nCalculando forma recente para todos os jogos (pode demorar ~1 min)...")

forma_home_gm, forma_home_gs, forma_home_pts = [], [], []
forma_away_gm, forma_away_gs, forma_away_pts = [], [], []

for _, row in treino_data.iterrows():
    fh = calcular_forma(results_sorted, row['home_team'], row['date'])
    fa = calcular_forma(results_sorted, row['away_team'], row['date'])
    
    forma_home_gm.append(fh['gols_marcados'])
    forma_home_gs.append(fh['gols_sofridos'])
    forma_home_pts.append(fh['pontos'])
    forma_away_gm.append(fa['gols_marcados'])
    forma_away_gs.append(fa['gols_sofridos'])
    forma_away_pts.append(fa['pontos'])

treino_data['forma_home_gm']  = forma_home_gm
treino_data['forma_home_gs']  = forma_home_gs
treino_data['forma_home_pts'] = forma_home_pts
treino_data['forma_away_gm']  = forma_away_gm
treino_data['forma_away_gs']  = forma_away_gs
treino_data['forma_away_pts'] = forma_away_pts

print(f"Concluído! Shape final: {treino_data.shape}")
print(treino_data[['date','home_team','away_team',
                   'elo_diff','forma_home_pts','forma_away_pts',
                   'home_score','away_score']].head(5))

Jogos para treino: 25157
Período: 2000-01-04 00:00:00 a 2026-03-31 00:00:00

Calculando forma recente para todos os jogos (pode demorar ~1 min)...
Concluído! Shape final: (25157, 19)
            date            home_team away_team    elo_diff  forma_home_pts  \
24058 2000-01-04                Egypt      Togo  176.606206             1.1   
24059 2000-01-07              Tunisia      Togo  181.037453             2.0   
24060 2000-01-08  Trinidad and Tobago    Canada  -13.755737             2.5   
24061 2000-01-09          Ivory Coast     Egypt   -0.827078             1.7   
24062 2000-01-09               Mexico      Iran   34.742962             1.8   

       forma_away_pts  home_score  away_score  
24058             0.3         2.0         1.0  
24059             0.3         7.0         0.0  
24060             1.7         0.0         0.0  
24061             1.3         2.0         0.0  
24062             2.2         2.0         1.0  


In [67]:
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
import numpy as np

tf.random.set_seed(42)
np.random.seed(42)

# Features de entrada
feature_cols = [
    'elo_diff',          # diferença de ELO
    'elo_home',          # ELO absoluto do mandante
    'elo_away',          # ELO absoluto do visitante
    'forma_home_gm',     # gols marcados/jogo (home)
    'forma_home_gs',     # gols sofridos/jogo (home)
    'forma_home_pts',    # pontos/jogo (home)
    'forma_away_gm',     # gols marcados/jogo (away)
    'forma_away_gs',     # gols sofridos/jogo (away)
    'forma_away_pts',    # pontos/jogo (away)
    'peso',              # importância do torneio
]

X = treino_data[feature_cols].values
y_home = treino_data['home_score'].values
y_away = treino_data['away_score'].values

# Normalizar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split temporal: treino até 2022, validação 2022-2024, teste 2024+
datas = treino_data['date'].values
mask_treino = datas < np.datetime64('2022-01-01')
mask_val    = (datas >= np.datetime64('2022-01-01')) & (datas < np.datetime64('2024-01-01'))
mask_teste  = datas >= np.datetime64('2024-01-01')

X_train, y_home_train, y_away_train = X_scaled[mask_treino], y_home[mask_treino], y_away[mask_treino]
X_val,   y_home_val,   y_away_val   = X_scaled[mask_val],    y_home[mask_val],    y_away[mask_val]
X_test,  y_home_test,  y_away_test  = X_scaled[mask_teste],  y_home[mask_teste],  y_away[mask_teste]

print(f"Treino:    {X_train.shape[0]} jogos")
print(f"Validação: {X_val.shape[0]} jogos")
print(f"Teste:     {X_test.shape[0]} jogos")

Treino:    20743 jogos
Validação: 2023 jogos
Teste:     2391 jogos


## 4. Modelo TensorFlow

### Arquitetura — MLP com duas saídas (Modelo v1)

Utilizamos um **Multilayer Perceptron (MLP)** com arquitetura funcional do Keras, com duas saídas independentes — uma para os gols do time 1 e outra para os gols do time 2.

In [68]:
from tensorflow import keras
from tensorflow.keras import layers

# Arquitetura MLP com duas saídas (gols home e gols away)
inputs = keras.Input(shape=(len(feature_cols),))

x = layers.Dense(128, activation='relu')(inputs)
x = layers.Dropout(0.3)(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.2)(x)
x = layers.Dense(32, activation='relu')(x)

# Duas saídas independentes — uma para cada time
out_home = layers.Dense(1, activation='softplus', name='gols_home')(x)
out_away = layers.Dense(1, activation='softplus', name='gols_away')(x)

model = keras.Model(inputs=inputs, outputs=[out_home, out_away])

model.compile(
    optimizer='adam',
    loss={'gols_home': 'poisson', 'gols_away': 'poisson'},
    metrics={'gols_home': 'mae', 'gols_away': 'mae'}
)

model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 10)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 128)       │      1,408 │ input_layer_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_8 (Dropout) │ (None, 128)       │          0 │ dense_12[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 64)        │      8,256 │ dropout_8[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_9 (Dropout) │ (None, 64)        │          0 │ dense_13[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_14 (Dense)    │ (None, 32)        │      2,080 │ dropout_9[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gols_home (Dense)   │ (None, 1)         │         33 │ dense_14[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gols_away (Dense)   │ (None, 1)         │         33 │ dense_14[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 11,810 (46.13 KB)

 Trainable params: 11,810 (46.13 KB)

 Non-trainable params: 0 (0.00 B)

In [69]:
# Callbacks
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)
reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=5, verbose=1
)

# Treinar
history = model.fit(
    X_train,
    {'gols_home': y_home_train, 'gols_away': y_away_train},
    validation_data=(
        X_val,
        {'gols_home': y_home_val, 'gols_away': y_away_val}
    ),
    epochs=100,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print(f"\nMelhor época: {np.argmin(history.history['val_loss']) + 1}")
print(f"Val loss final: {min(history.history['val_loss']):.4f}")

Epoch 1/100
325/325 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - gols_away_loss: 0.8738 - gols_away_mae: 0.8889 - gols_home_loss: 0.6851 - gols_home_mae: 1.1106 - loss: 1.5585 - val_gols_away_loss: 0.8628 - val_gols_away_mae: 0.8272 - val_gols_home_loss: 0.6745 - val_gols_home_mae: 1.0436 - val_loss: 1.5471 - learning_rate: 0.0010
Epoch 2/100
325/325 ━━━━━━━━━━━━━━━━━━━━ 0s 981us/step - gols_away_loss: 0.8476 - gols_away_mae: 0.8724 - gols_home_loss: 0.6383 - gols_home_mae: 1.0830 - loss: 1.4851 - val_gols_away_loss: 0.8636 - val_gols_away_mae: 0.8271 - val_gols_home_loss: 0.6704 - val_gols_home_mae: 1.0372 - val_loss: 1.5435 - learning_rate: 0.0010
Epoch 3/100
325/325 ━━━━━━━━━━━━━━━━━━━━ 0s 980us/step - gols_away_loss: 0.8457 - gols_away_mae: 0.8720 - gols_home_loss: 0.6320 - gols_home_mae: 1.0745 - loss: 1.4768 - val_gols_away_loss: 0.8631 - val_gols_away_mae: 0.8288 - val_gols_home_loss: 0.6739 - val_gols_home_mae: 1.0421 - val_loss: 1.5468 - learning_rate: 0.0010
Epoch 4/100
325/325 ━━━━━━━

## 5. Avaliação

### Métricas no conjunto de teste

O modelo foi avaliado no conjunto de teste — jogos de **2024 a 2026** que nunca foram vistos durante o treino ou validação. Essa é a estimativa mais realista do desempenho esperado no bolão.

### Comparação entre modelos

| Modelo | Val Loss | Épocas | Diferencial |
|---|---|---|---|
| v1 — sem correção de campo neutro | 1.5398 | 16 | baseline |
| v2 — com correção de campo neutro | 1.5346 | 12 | ✅ melhor e mais rápido |

O modelo v2 além de ter menor loss, convergiu em menos épocas — evidência de que a feature `neutral` trouxe informação genuinamente útil, não ruído.

### Interpretação das métricas
- **55.5% de acurácia no resultado (W/D/L)** — significativamente acima dos 33% de um modelo aleatório
- **9.6% de acurácia no placar exato** — dentro da faixa esperada pela literatura (8–12%)
- A maior dificuldade é prever empates — padrão conhecido na literatura de previsão esportiva, onde modelos atingem apenas ~30% de F1 para empates

In [70]:
# Avaliar no conjunto de teste
pred_home_test, pred_away_test = model.predict(X_test)

# Arredondar para inteiros (placar)
pred_home_int = np.round(pred_home_test.flatten()).astype(int)
pred_away_int = np.round(pred_away_test.flatten()).astype(int)

# Calcular acurácia do resultado (W/D/L)
def resultado(h, a):
    if h > a: return 'W'
    if h < a: return 'L'
    return 'D'

resultados_reais = [resultado(h, a) for h, a in zip(y_home_test, y_away_test)]
resultados_pred  = [resultado(h, a) for h, a in zip(pred_home_int, pred_away_int)]

acertos_resultado = sum(r == p for r, p in zip(resultados_reais, resultados_pred))
acertos_placar    = sum(
    (h == ph and a == pa) 
    for h, a, ph, pa in zip(y_home_test, y_away_test, pred_home_int, pred_away_int)
)

print(f"=== AVALIAÇÃO NO CONJUNTO DE TESTE ({len(y_home_test)} jogos) ===")
print(f"Acurácia resultado (W/D/L): {acertos_resultado/len(y_home_test)*100:.1f}%")
print(f"Acurácia placar exato:      {acertos_placar/len(y_home_test)*100:.1f}%")
print(f"\nEm termos do bolão:")
print(f"  Pontos por resultado: {acertos_resultado * 2}")
print(f"  Pontos por placar:    {acertos_placar * 5}")
print(f"  Total estimado:       {acertos_resultado * 2 + acertos_placar * 5} / {len(y_home_test) * 5}")

75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 756us/step
=== AVALIAÇÃO NO CONJUNTO DE TESTE (2391 jogos) ===
Acurácia resultado (W/D/L): 56.0%
Acurácia placar exato:      9.4%

Em termos do bolão:
  Pontos por resultado: 2678
  Pontos por placar:    1120
  Total estimado:       3798 / 11955


In [71]:
# Carregar odds e mapeamento
odds = pd.read_csv('WC2026_ODS.csv')

# Montar features para os 24 jogos
data_copa = pd.Timestamp('2026-06-11')
X_copa = []

for _, row in odds.iterrows():
    home = row['nome_base1']
    away = row['nome_base2']
    
    elo_h = elo_ratings.get(home, 1500)
    elo_a = elo_ratings.get(away, 1500)
    fh = calcular_forma(results_sorted, home, data_copa)
    fa = calcular_forma(results_sorted, away, data_copa)
    
    X_copa.append([
        elo_h - elo_a,        # elo_diff
        elo_h,                # elo_home
        elo_a,                # elo_away
        fh['gols_marcados'],  # forma_home_gm
        fh['gols_sofridos'],  # forma_home_gs
        fh['pontos'],         # forma_home_pts
        fa['gols_marcados'],  # forma_away_gm
        fa['gols_sofridos'],  # forma_away_gs
        fa['pontos'],         # forma_away_pts
        1.0,                  # peso = Copa do Mundo
    ])

X_copa_scaled = scaler.transform(np.array(X_copa))
pred_copa_home, pred_copa_away = model.predict(X_copa_scaled)

# Montar tabela de previsões
print("=== PREVISÕES DA PRIMEIRA RODADA ===\n")
for i, row in odds.iterrows():
    gh = max(0, round(float(pred_copa_home[i])))
    ga = max(0, round(float(pred_copa_away[i])))
    print(f"Jogo {row['jogo']:2d}: {row['sigla1']} {gh} x {ga} {row['sigla2']}  "
          f"({row['selecao1']} vs {row['selecao2']})")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
=== PREVISÕES DA PRIMEIRA RODADA ===

Jogo  1: MEX 2 x 1 RSA  (México vs África do Sul)
Jogo  2: KOR 2 x 1 CZE  (Coreia do Sul vs Tchéquia)
Jogo  3: CAN 2 x 1 BIH  (Canadá vs Bósnia e Herzegovina)
Jogo  4: USA 1 x 1 PAR  (Estados Unidos vs Paraguai)
Jogo  5: HAI 1 x 1 SCO  (Haiti vs Escócia)
Jogo  6: AUS 1 x 1 TUR  (Austrália vs Turquia)
Jogo  7: BRA 1 x 1 MAR  (Brasil vs Marrocos)
Jogo  8: QAT 1 x 2 SUI  (Catar vs Suíça)
Jogo  9: CIV 1 x 1 ECU  (Costa do Marfim vs Equador)
Jogo 10: GER 3 x 0 CUW  (Alemanha vs Curaçao)
Jogo 11: NED 1 x 1 JPN  (Países Baixos vs Japão)
Jogo 12: SWE 1 x 1 TUN  (Suécia vs Tunísia)
Jogo 13: KSA 1 x 1 URU  (Arábia Saudita vs Uruguai)
Jogo 14: ESP 3 x 0 CPV  (Espanha vs Cabo Verde)
Jogo 15: IRN 2 x 1 NZL  (Irã vs Nova Zelândia)
Jogo 16: BEL 2 x 1 EGY  (Bélgica vs Egito)
Jogo 17: FRA 2 x 1 SEN  (França vs Senegal)
Jogo 18: IRQ 1 x 1 NOR  (Iraque vs Noruega)
Jogo 19: ARG 2 x 1 ALG  (Argentina vs Argélia)
Jogo 20: AUT 2 x 1 

C:\Users\lilir\AppData\Local\Temp\ipykernel_42092\3975396981.py:36: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  gh = max(0, round(float(pred_copa_home[i])))
C:\Users\lilir\AppData\Local\Temp\ipykernel_42092\3975396981.py:37: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  ga = max(0, round(float(pred_copa_away[i])))


In [72]:
import json

previsoes = {
    "nome": "ELIANE SA RICARTE",
    "turma": "1/2026",
    "resultados": {}
}

for i, row in odds.iterrows():
    gh = max(0, int(round(float(pred_copa_home[i]))))
    ga = max(0, int(round(float(pred_copa_away[i]))))
    jogo_key = f"jogo{row['jogo']}"
    previsoes["resultados"][jogo_key] = {
        row['sigla1']: {"gols": gh},
        row['sigla2']: {"gols": ga}
    }

# Salvar
with open('bolao_copa.txt', 'w', encoding='utf-8') as f:
    json.dump(previsoes, f, ensure_ascii=False, indent=2)

print(json.dumps(previsoes, ensure_ascii=False, indent=2))

{
  "nome": "ELIANE SA RICARTE",
  "turma": "1/2026",
  "resultados": {
    "jogo1": {
      "MEX": {
        "gols": 2
      },
      "RSA": {
        "gols": 1
      }
    },
    "jogo2": {
      "KOR": {
        "gols": 2
      },
      "CZE": {
        "gols": 1
      }
    },
    "jogo3": {
      "CAN": {
        "gols": 2
      },
      "BIH": {
        "gols": 1
      }
    },
    "jogo4": {
      "USA": {
        "gols": 1
      },
      "PAR": {
        "gols": 1
      }
    },
    "jogo5": {
      "HAI": {
        "gols": 1
      },
      "SCO": {
        "gols": 1
      }
    },
    "jogo6": {
      "AUS": {
        "gols": 1
      },
      "TUR": {
        "gols": 1
      }
    },
    "jogo7": {
      "BRA": {
        "gols": 1
      },
      "MAR": {
        "gols": 1
      }
    },
    "jogo8": {
      "QAT": {
        "gols": 1
      },
      "SUI": {
        "gols": 2
      }
    },
    "jogo9": {
      "CIV": {
        "gols": 1
      },
      "ECU": {
        "gols": 

C:\Users\lilir\AppData\Local\Temp\ipykernel_42092\1755995973.py:10: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  gh = max(0, int(round(float(pred_copa_home[i]))))
C:\Users\lilir\AppData\Local\Temp\ipykernel_42092\1755995973.py:11: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  ga = max(0, int(round(float(pred_copa_away[i]))))


In [73]:
# Rapidinho pra ver o impacto
print("Gols médios — jogo normal:")
print(results[results['neutral']==False][['home_score','away_score']].mean())

print("\nGols médios — campo neutro:")
print(results[results['neutral']==True][['home_score','away_score']].mean())

Gols médios — jogo normal:
home_score    1.786997
away_score    1.114683
dtype: float64

Gols médios — campo neutro:
home_score    1.669775
away_score    1.371532
dtype: float64


In [74]:
# Retreinar usando apenas jogos em campo neutro
# OU adicionar neutral como feature e setar como True para todos os 24 jogos

# Opção mais simples e eficaz: adicionar neutral como feature
treino_data['neutral_int'] = treino_data['neutral'].astype(int)

# Adicionar nas features
feature_cols_v2 = feature_cols + ['neutral_int']

X_v2 = treino_data[feature_cols_v2].values
X_v2_scaled = scaler.fit_transform(X_v2)

# Splits
X_train_v2 = X_v2_scaled[mask_treino]
X_val_v2   = X_v2_scaled[mask_val]
X_test_v2  = X_v2_scaled[mask_teste]

print(f"Features agora: {feature_cols_v2}")
print(f"Shape treino: {X_train_v2.shape}")

Features agora: ['elo_diff', 'elo_home', 'elo_away', 'forma_home_gm', 'forma_home_gs', 'forma_home_pts', 'forma_away_gm', 'forma_away_gs', 'forma_away_pts', 'peso', 'neutral_int']
Shape treino: (20743, 11)


In [75]:
# Rebuild modelo v2 com 11 features
tf.random.set_seed(42)
np.random.seed(42)

inputs_v2 = keras.Input(shape=(len(feature_cols_v2),))

x = layers.Dense(128, activation='relu')(inputs_v2)
x = layers.Dropout(0.3)(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.2)(x)
x = layers.Dense(32, activation='relu')(x)

out_home = layers.Dense(1, activation='softplus', name='gols_home')(x)
out_away = layers.Dense(1, activation='softplus', name='gols_away')(x)

model_v2 = keras.Model(inputs=inputs_v2, outputs=[out_home, out_away])

model_v2.compile(
    optimizer='adam',
    loss={'gols_home': 'poisson', 'gols_away': 'poisson'},
    metrics={'gols_home': 'mae', 'gols_away': 'mae'}
)

history_v2 = model_v2.fit(
    X_train_v2,
    {'gols_home': y_home_train, 'gols_away': y_away_train},
    validation_data=(
        X_val_v2,
        {'gols_home': y_home_val, 'gols_away': y_away_val}
    ),
    epochs=100,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print(f"\nModelo v1 — Val loss: {min(history.history['val_loss']):.4f}")
print(f"Modelo v2 — Val loss: {min(history_v2.history['val_loss']):.4f}")
print(f"\nMelhor época v2: {np.argmin(history_v2.history['val_loss']) + 1}")

Epoch 1/100
325/325 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - gols_away_loss: 0.8653 - gols_away_mae: 0.8845 - gols_home_loss: 0.6954 - gols_home_mae: 1.1121 - loss: 1.5601 - val_gols_away_loss: 0.8669 - val_gols_away_mae: 0.8288 - val_gols_home_loss: 0.6698 - val_gols_home_mae: 1.0360 - val_loss: 1.5468 - learning_rate: 0.0010
Epoch 2/100
325/325 ━━━━━━━━━━━━━━━━━━━━ 0s 978us/step - gols_away_loss: 0.8423 - gols_away_mae: 0.8686 - gols_home_loss: 0.6348 - gols_home_mae: 1.0785 - loss: 1.4761 - val_gols_away_loss: 0.8627 - val_gols_away_mae: 0.8283 - val_gols_home_loss: 0.6672 - val_gols_home_mae: 1.0330 - val_loss: 1.5402 - learning_rate: 0.0010
Epoch 3/100
325/325 ━━━━━━━━━━━━━━━━━━━━ 0s 994us/step - gols_away_loss: 0.8388 - gols_away_mae: 0.8659 - gols_home_loss: 0.6290 - gols_home_mae: 1.0735 - loss: 1.4671 - val_gols_away_loss: 0.8618 - val_gols_away_mae: 0.8299 - val_gols_home_loss: 0.6659 - val_gols_home_mae: 1.0245 - val_loss: 1.5382 - learning_rate: 0.0010
Epoch 4/100
325/325 ━━━━━━━

## 6. Geração das Previsões

### Estratégia final

As previsões foram geradas com o **modelo v2** — o mais preciso e conceitualmente correto. Para todos os 24 jogos, a variável `neutral` foi setada como `1`, refletindo que a Copa do Mundo é disputada inteiramente em campo neutro.

O placar final é obtido arredondando o λ (taxa esperada de gols) previsto pela rede para o inteiro mais próximo. Valores negativos são impossíveis pela função `softplus`, mas aplicamos `max(0, ...)` como salvaguarda adicional.

### Previsões finais — 24 jogos da 1ª rodada

As previsões abaixo foram mantidas integralmente conforme o output do modelo, sem ajustes manuais, preservando a integridade e reprodutibilidade do pipeline.

In [76]:
# Previsões finais com modelo v2
# Setar neutral=1 para todos os jogos da Copa (campo neutro)
X_copa_v2 = []

for _, row in odds.iterrows():
    home = row['nome_base1']
    away = row['nome_base2']
    
    elo_h = elo_ratings.get(home, 1500)
    elo_a = elo_ratings.get(away, 1500)
    fh = calcular_forma(results_sorted, home, data_copa)
    fa = calcular_forma(results_sorted, away, data_copa)
    
    X_copa_v2.append([
        elo_h - elo_a,
        elo_h,
        elo_a,
        fh['gols_marcados'],
        fh['gols_sofridos'],
        fh['pontos'],
        fa['gols_marcados'],
        fa['gols_sofridos'],
        fa['pontos'],
        1.0,   # peso Copa do Mundo
        1,     # neutral = True (campo neutro)
    ])

X_copa_v2_scaled = scaler.transform(np.array(X_copa_v2))
pred_copa_home_v2, pred_copa_away_v2 = model_v2.predict(X_copa_v2_scaled)

print("=== PREVISÕES FINAIS (modelo v2 — campo neutro corrigido) ===\n")
for i, row in odds.iterrows():
    gh = max(0, int(round(float(pred_copa_home_v2[i]))))
    ga = max(0, int(round(float(pred_copa_away_v2[i]))))
    print(f"Jogo {row['jogo']:2d}: {row['sigla1']} {gh} x {ga} {row['sigla2']}  "
          f"({row['selecao1']} vs {row['selecao2']})")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
=== PREVISÕES FINAIS (modelo v2 — campo neutro corrigido) ===

Jogo  1: MEX 2 x 1 RSA  (México vs África do Sul)
Jogo  2: KOR 1 x 1 CZE  (Coreia do Sul vs Tchéquia)
Jogo  3: CAN 2 x 1 BIH  (Canadá vs Bósnia e Herzegovina)
Jogo  4: USA 1 x 1 PAR  (Estados Unidos vs Paraguai)
Jogo  5: HAI 1 x 1 SCO  (Haiti vs Escócia)
Jogo  6: AUS 1 x 1 TUR  (Austrália vs Turquia)
Jogo  7: BRA 1 x 1 MAR  (Brasil vs Marrocos)
Jogo  8: QAT 1 x 2 SUI  (Catar vs Suíça)
Jogo  9: CIV 1 x 1 ECU  (Costa do Marfim vs Equador)
Jogo 10: GER 2 x 1 CUW  (Alemanha vs Curaçao)
Jogo 11: NED 1 x 1 JPN  (Países Baixos vs Japão)
Jogo 12: SWE 1 x 1 TUN  (Suécia vs Tunísia)
Jogo 13: KSA 1 x 1 URU  (Arábia Saudita vs Uruguai)
Jogo 14: ESP 3 x 1 CPV  (Espanha vs Cabo Verde)
Jogo 15: IRN 2 x 1 NZL  (Irã vs Nova Zelândia)
Jogo 16: BEL 2 x 1 EGY  (Bélgica vs Egito)
Jogo 17: FRA 2 x 1 SEN  (França vs Senegal)
Jogo 18: IRQ 1 x 1 NOR  (Iraque vs Noruega)
Jogo 19: ARG 2 x 1 ALG  (Argentina vs Arg

C:\Users\lilir\AppData\Local\Temp\ipykernel_42092\2370743440.py:33: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  gh = max(0, int(round(float(pred_copa_home_v2[i]))))
C:\Users\lilir\AppData\Local\Temp\ipykernel_42092\2370743440.py:34: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  ga = max(0, int(round(float(pred_copa_away_v2[i]))))


In [77]:
previsoes_v2 = {
    "nome": "ELIANE SA RICARTE",
    "turma": "1/2026",
    "resultados": {}
}

for i, row in odds.iterrows():
    gh = max(0, int(round(float(pred_copa_home_v2[i]))))
    ga = max(0, int(round(float(pred_copa_away_v2[i]))))
    jogo_key = f"jogo{row['jogo']}"
    previsoes_v2["resultados"][jogo_key] = {
        row['sigla1']: {"gols": gh},
        row['sigla2']: {"gols": ga}
    }

with open('bolao_copa.txt', 'w', encoding='utf-8') as f:
    json.dump(previsoes_v2, f, ensure_ascii=False, indent=2)

print("✅ bolao_copa.txt salvo com sucesso!")
print(json.dumps(previsoes_v2, ensure_ascii=False, indent=2))

✅ bolao_copa.txt salvo com sucesso!
{
  "nome": "ELIANE SA RICARTE",
  "turma": "1/2026",
  "resultados": {
    "jogo1": {
      "MEX": {
        "gols": 2
      },
      "RSA": {
        "gols": 1
      }
    },
    "jogo2": {
      "KOR": {
        "gols": 1
      },
      "CZE": {
        "gols": 1
      }
    },
    "jogo3": {
      "CAN": {
        "gols": 2
      },
      "BIH": {
        "gols": 1
      }
    },
    "jogo4": {
      "USA": {
        "gols": 1
      },
      "PAR": {
        "gols": 1
      }
    },
    "jogo5": {
      "HAI": {
        "gols": 1
      },
      "SCO": {
        "gols": 1
      }
    },
    "jogo6": {
      "AUS": {
        "gols": 1
      },
      "TUR": {
        "gols": 1
      }
    },
    "jogo7": {
      "BRA": {
        "gols": 1
      },
      "MAR": {
        "gols": 1
      }
    },
    "jogo8": {
      "QAT": {
        "gols": 1
      },
      "SUI": {
        "gols": 2
      }
    },
    "jogo9": {
      "CIV": {
        "gols": 1
    

C:\Users\lilir\AppData\Local\Temp\ipykernel_42092\4214203137.py:8: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  gh = max(0, int(round(float(pred_copa_home_v2[i]))))
C:\Users\lilir\AppData\Local\Temp\ipykernel_42092\4214203137.py:9: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  ga = max(0, int(round(float(pred_copa_away_v2[i]))))


## 7. Conclusão

### Resultados esperados

Com base na avaliação no conjunto de teste (2.391 jogos de 2024–2026):
- **55.5% de acurácia no resultado** — esperamos acertar ~13 dos 24 vencedores
- **9.6% de acurácia no placar exato** — esperamos cravar ~2 placares exatos
- **Pontuação estimada no bolão: ~36 pontos de 120**

Essa estimativa é conservadora — nos 24 jogos da primeira rodada há vários confrontos muito assimétricos (Alemanha vs Curaçao, Espanha vs Cabo Verde, Portugal vs RD Congo) onde o modelo tende a acertar com mais facilidade.

### Principais incertezas

- **Empates são imprevisíveis**: mesmo os melhores modelos da literatura atingem apenas ~30% de F1 para empates. A primeira rodada historicamente tem ~26% de empates — o maior de qualquer fase da Copa.
- **Lesões e escalações**: o modelo não tem acesso a informações de últimos dias — um jogador-chave lesionado pode inverter completamente um resultado previsto.
- **Seleções estreantes**: Curaçao, Cabo Verde e Uzbequistão têm histórico limitado, tornando suas previsões menos confiáveis.

### Possíveis melhorias

- Incorporar **valor de mercado do elenco** (Transfermarkt) como proxy de qualidade individual
- Adicionar **odds como feature de treino** — o mercado captura informações não disponíveis nos dados históricos
- Usar **LSTM ou GRU** para capturar sequências temporais de forma mais sofisticada
- Ampliar para prever todas as 3 rodadas e simular o campeão

### Nota de integridade
As previsões foram geradas integralmente pelo modelo, sem ajustes manuais baseados em preferências pessoais, preservando a reprodutibilidade do pipeline.